<a href="https://colab.research.google.com/github/shonbo1221-collab/DRL_FinalProjectProposal/blob/main/train_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TWSE PPO + SMC Training Notebook

This notebook version mirrors `train_pipeline.py` and splits the training flow into editable steps.

## 1. Imports

### Colab dependency setup

Run this cell once after starting or restarting a Google Colab runtime.

In [5]:
%pip install -q stable-baselines3==2.8.0 gymnasium==1.2.3 yfinance==1.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 2.2 MB/s eta 0:00:00


In [6]:
import os
from dataclasses import dataclass

import gymnasium as gym
import numpy as np
import pandas as pd
import torch
import yfinance as yf
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 2. Shared Trading Module

The definitions below are embedded from `twse_pipeline_common.py` so this notebook can run without importing any local project module.

In [7]:
BROKERAGE_FEE = 0.001425 * 0.6
PRICE_LIMIT_UP = 1.10
SHARPE_WINDOW = 20
EPS = 1e-8


@dataclass(frozen=True)
class ModelConfig:
    name: str
    tickers: tuple[str, ...]
    model_path: str
    mode: str


PAIR_CONFIG = ModelConfig(
    name="pair_0050_2330",
    tickers=("0050.TW", "2330.TW"),
    model_path="model/save_colab/ppo_model_pair.zip",
    mode="pair",
)

BASKET_CONFIG = ModelConfig(
    name="basket_0050_2330_2412",
    tickers=("0050.TW", "2330.TW", "2412.TW"),
    model_path="model/save_colab/ppo_model_basket.zip",
    mode="basket",
)

SELL_TAX = {
    "0050.TW": 0.001,
    "2330.TW": 0.003,
    "2412.TW": 0.003,
}

In [8]:
def download_ohlcv(tickers, start_date, end_date):
    frames = {}
    for ticker in tickers:
        df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)
        if df.empty:
            raise ValueError(f"No data downloaded for {ticker}")
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.reset_index()
        df = df.rename(
            columns={
                "Date": "date",
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close",
                "Adj Close": "adj_close",
                "Volume": "volume",
            }
        )
        required = ["date", "open", "high", "low", "close", "volume"]
        frames[ticker] = df[required].dropna().reset_index(drop=True)
    return frames


def add_smc_features(df):
    out = df.copy()
    high_20 = out["high"].rolling(20).max()
    low_20 = out["low"].rolling(20).min()
    dealing_range = (high_20 - low_20).replace(0, np.nan)
    out["PD_Pos"] = ((out["close"] - low_20) / dealing_range).clip(0.0, 1.0)

    ret_3d = out["close"].pct_change(3)
    bearish_candle = out["close"] < out["open"]
    ob_level = np.nan
    ob_levels = []
    for i in range(len(out)):
        if i >= 3 and ret_3d.iloc[i] > 0.03:
            start = max(0, i - 20)
            for j in range(i - 3, start - 1, -1):
                if bearish_candle.iloc[j]:
                    ob_level = out["high"].iloc[j]
                    break
        ob_levels.append(ob_level)
    out["ob_level"] = pd.Series(ob_levels, index=out.index).ffill()
    out["OB_Dist"] = ((out["close"] - out["ob_level"]) / out["close"]).replace([np.inf, -np.inf], np.nan)

    bullish_fvg = out["low"] > out["high"].shift(2)
    bearish_fvg = out["high"] < out["low"].shift(2)
    bull_low = out["high"].shift(2)
    bull_high = out["low"]
    bear_low = out["high"]
    bear_high = out["low"].shift(2)

    unfilled = np.zeros(len(out), dtype=np.float32)
    active_gaps = []
    for i in range(len(out)):
        still_active = []
        for gap in active_gaps:
            touches_gap = out["low"].iloc[i] <= gap["high"] and out["high"].iloc[i] >= gap["low"]
            if not touches_gap:
                still_active.append(gap)
        active_gaps = still_active

        if bullish_fvg.iloc[i]:
            active_gaps.append({"low": bull_low.iloc[i], "high": bull_high.iloc[i], "direction": "bull"})
        if bearish_fvg.iloc[i]:
            active_gaps.append({"low": bear_low.iloc[i], "high": bear_high.iloc[i], "direction": "bear"})

        unfilled[i] = 1.0 if active_gaps else 0.0
    out["FVG_Signal"] = unfilled

    out[["PD_Pos", "OB_Dist", "FVG_Signal"]] = out[["PD_Pos", "OB_Dist", "FVG_Signal"]].fillna(
        {"PD_Pos": 0.5, "OB_Dist": 0.0, "FVG_Signal": 0.0}
    )
    return out

In [9]:
def build_feature_frame(raw_frames, config):
    feature_frames = []
    for ticker in config.tickers:
        df = add_smc_features(raw_frames[ticker])
        prefix = ticker.replace(".", "_")
        cols = ["date", "open", "high", "low", "close", "volume", "PD_Pos", "OB_Dist", "FVG_Signal"]
        renamed = df[cols].rename(columns={col: f"{prefix}_{col}" for col in cols if col != "date"})
        feature_frames.append(renamed)

    merged = feature_frames[0]
    for frame in feature_frames[1:]:
        merged = pd.merge(merged, frame, on="date", how="inner")

    pairs = [(config.tickers[0], config.tickers[1])] if config.mode == "pair" else [
        (config.tickers[i], config.tickers[j])
        for i in range(len(config.tickers))
        for j in range(i + 1, len(config.tickers))
    ]
    for left_ticker, right_ticker in pairs:
        left = left_ticker.replace(".", "_")
        right = right_ticker.replace(".", "_")
        col = f"Spread_ZScore_{left}_{right}"
        spread = np.log(merged[f"{left}_close"]) - np.log(merged[f"{right}_close"])
        mean = spread.rolling(20).mean()
        std = spread.rolling(20).std().replace(0, np.nan)
        merged[col] = ((spread - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if config.mode == "pair":
        left = config.tickers[0].replace(".", "_")
        right = config.tickers[1].replace(".", "_")
        merged["Spread_ZScore"] = merged[f"Spread_ZScore_{left}_{right}"]
    else:
        merged["Spread_ZScore"] = 0.0

    return merged.dropna().reset_index(drop=True)

In [10]:
class TWTradingEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, feature_df, config, initial_balance=1_000_000.0, max_position=0.95):
        super().__init__()
        self.df = feature_df.reset_index(drop=True)
        self.config = config
        self.initial_balance = float(initial_balance)
        self.max_position = float(max_position)
        self.asset_prefixes = [ticker.replace(".", "_") for ticker in config.tickers]
        self.feature_cols = self._make_feature_cols()

        obs_dim = len(self.feature_cols) + len(self.config.tickers) + 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        action_dim = len(self.config.tickers) if self.config.mode == "basket" else 1
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(action_dim,), dtype=np.float32)

    def _make_feature_cols(self):
        cols = []
        for prefix in self.asset_prefixes:
            cols.extend([f"{prefix}_PD_Pos", f"{prefix}_OB_Dist", f"{prefix}_FVG_Signal"])
        if self.config.mode == "pair":
            cols.append("Spread_ZScore")
        elif self.config.mode == "basket":
            cols.extend([col for col in self.df.columns if col.startswith("Spread_ZScore_")])
        return cols

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.cash = self.initial_balance
        self.shares = np.zeros(len(self.config.tickers), dtype=np.float64)
        self.net_worth = self.initial_balance
        self.max_net_worth = self.initial_balance
        self.prev_drawdown = 0.0
        self.portfolio_returns = []
        self.history = []
        return self._get_obs(), {}

    def _prices(self, step=None):
        idx = self.current_step if step is None else step
        return np.array([self.df.loc[idx, f"{prefix}_close"] for prefix in self.asset_prefixes], dtype=np.float64)

    def _prev_closes(self):
        prev_step = max(0, self.current_step - 1)
        return np.array([self.df.loc[prev_step, f"{prefix}_close"] for prefix in self.asset_prefixes], dtype=np.float64)

    def _get_weights(self, prices=None, net_worth=None):
        prices = self._prices() if prices is None else prices
        net_worth = self.net_worth if net_worth is None else net_worth
        if net_worth <= 0:
            return np.zeros(len(self.config.tickers), dtype=np.float64)
        return (self.shares * prices) / net_worth

    def _get_obs(self):
        market_features = self.df.loc[self.current_step, self.feature_cols].to_numpy(dtype=np.float32)
        prices = self._prices()
        net_worth = self.cash + float(np.dot(self.shares, prices))
        weights = self._get_weights(prices, net_worth).astype(np.float32)
        cash_ratio = np.array([self.cash / max(net_worth, EPS)], dtype=np.float32)
        return np.concatenate([market_features, weights, cash_ratio]).astype(np.float32)

    def _target_weights(self, raw_action):
        action_array = np.asarray(raw_action, dtype=np.float64).reshape(-1)
        if self.config.mode == "pair":
            action = float(np.clip(action_array[0], -1.0, 1.0))
            weights = np.zeros(2, dtype=np.float64)
            if action >= 0:
                weights[0] = action * self.max_position
            else:
                weights[1] = -action * self.max_position
            return weights

        if self.config.mode == "basket":
            scores = np.clip(action_array[: len(self.config.tickers)], 0.0, 1.0)
            if scores.sum() <= EPS:
                return np.zeros(len(self.config.tickers), dtype=np.float64)
            return (scores / scores.sum()) * self.max_position

        action = float(np.clip(action_array[0], -1.0, 1.0))
        return np.array([max(0.0, action) * self.max_position], dtype=np.float64)

    def _execute_rebalance(self, target_weights, prices):
        prev_prices = self._prev_closes()
        net_worth_before_trade = self.cash + float(np.dot(self.shares, prices))
        current_values = self.shares * prices
        target_values = target_weights * net_worth_before_trade
        value_diffs = target_values - current_values
        transaction_cost = 0.0
        failed_orders = 0

        for idx, value_diff in enumerate(value_diffs):
            ticker = self.config.tickers[idx]
            price = prices[idx]
            limit_up = prev_prices[idx] * PRICE_LIMIT_UP

            if abs(value_diff) > EPS and price >= limit_up:
                failed_orders += 1
                continue

            if value_diff > 0:
                buy_value = min(value_diff, max(0.0, self.cash / (1.0 + BROKERAGE_FEE)))
                fee = buy_value * BROKERAGE_FEE
                self.cash -= buy_value + fee
                self.shares[idx] += buy_value / price
                transaction_cost += fee
            elif value_diff < 0:
                sell_value = min(-value_diff, self.shares[idx] * price)
                fee = sell_value * BROKERAGE_FEE
                tax = sell_value * SELL_TAX[ticker]
                self.cash += sell_value - fee - tax
                self.shares[idx] -= sell_value / price
                transaction_cost += fee + tax

        return transaction_cost, failed_orders

    def step(self, action):
        raw_action_array = np.asarray(action, dtype=np.float64).reshape(-1)
        raw_action = float(raw_action_array[0])
        prices = self._prices()
        prev_net_worth = self.net_worth
        target_weights = self._target_weights(raw_action_array)
        transaction_cost, failed_orders = self._execute_rebalance(target_weights, prices)

        self.net_worth = self.cash + float(np.dot(self.shares, prices))
        pnl = self.net_worth - prev_net_worth
        pnl_rate = pnl / max(prev_net_worth, EPS)
        self.portfolio_returns.append(pnl_rate)

        recent = np.array(self.portfolio_returns[-SHARPE_WINDOW:], dtype=np.float64)
        if len(recent) > 1 and np.std(recent) > EPS:
            sharpe_adjustment = max(0.0, np.mean(recent) / (np.std(recent) + EPS) * np.sqrt(252))
        else:
            sharpe_adjustment = 1.0

        self.max_net_worth = max(self.max_net_worth, self.net_worth)
        drawdown = (self.max_net_worth - self.net_worth) / max(self.max_net_worth, EPS)
        drawdown_penalty = max(0.0, drawdown - self.prev_drawdown)
        self.prev_drawdown = drawdown

        cost_rate = transaction_cost / max(prev_net_worth, EPS)
        reward = (pnl_rate * sharpe_adjustment) - cost_rate - (2.0 * drawdown_penalty)

        weights = self._get_weights(prices, self.net_worth)
        self.history.append(
            {
                "date": self.df.loc[self.current_step, "date"],
                "raw_action": raw_action,
                "net_worth": self.net_worth,
                "pnl": pnl,
                "pnl_rate": pnl_rate,
                "transaction_cost": transaction_cost,
                "drawdown": drawdown,
                "reward": reward,
                "failed_orders": failed_orders,
                **{f"raw_action_{ticker}": raw_action_array[i] for i, ticker in enumerate(self.config.tickers) if i < len(raw_action_array)},
                **{f"weight_{ticker}": weights[i] for i, ticker in enumerate(self.config.tickers)},
            }
        )

        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        truncated = False
        obs = np.zeros(self.observation_space.shape, dtype=np.float32) if terminated else self._get_obs()
        info = {"net_worth": self.net_worth, "drawdown": drawdown, "transaction_cost": transaction_cost}
        return obs, float(reward), terminated, truncated, info

In [11]:
def calculate_metrics(history_df, initial_balance=1_000_000.0):
    if history_df.empty:
        return {"cumulative_return": 0.0, "sharpe_ratio": 0.0, "max_drawdown": 0.0}
    net_worth = history_df["net_worth"].astype(float)
    cumulative_return = (net_worth.iloc[-1] / initial_balance) - 1.0
    returns = net_worth.pct_change().dropna()
    sharpe = 0.0
    if len(returns) > 1 and returns.std() > EPS:
        sharpe = (returns.mean() / returns.std()) * np.sqrt(252)
    roll_max = net_worth.cummax()
    drawdown = (net_worth - roll_max) / roll_max
    return {
        "cumulative_return": float(cumulative_return),
        "sharpe_ratio": float(sharpe),
        "max_drawdown": float(drawdown.min()),
    }


def prepare_dataset(config, start_date, end_date):
    raw_frames = download_ohlcv(config.tickers, start_date, end_date)
    return build_feature_frame(raw_frames, config)


def ensure_model_dir(path):
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)

## 3. Training Settings

Adjust these values before running the training cells. The original script uses `500_000` timesteps and `cuda:0`.

In [12]:
TRAIN_START = "2018-01-01"
TRAIN_END = "2024-12-31"
TOTAL_TIMESTEPS = 500_000
INITIAL_BALANCE = 1_000_000.0
DEVICE = "cuda:0"

# Train both models by default, same as train_pipeline.py.
# You can change this to [PAIR_CONFIG] or [BASKET_CONFIG] for a shorter run.
CONFIGS_TO_TRAIN = [PAIR_CONFIG, BASKET_CONFIG]

## 4. CUDA Check

The original training script requires a CUDA GPU. Run this cell before preparing data or training.

In [14]:
def require_cuda():
    # Use global DEVICE to modify it if necessary
    global DEVICE
    if not torch.cuda.is_available():
        if DEVICE.startswith("cuda"):
            print("Warning: CUDA GPU is not available. Switching to CPU for training.")
            DEVICE = "cpu"
        else: # If DEVICE was already 'cpu' or something else, and CUDA is not available, just proceed.
            pass

    if DEVICE.startswith("cuda"):
        torch.set_float32_matmul_precision("high")
        torch.backends.cudnn.benchmark = True


require_cuda()
print(f"Using fixed training device: {DEVICE}")
if DEVICE.startswith("cuda"):
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU.")

Using fixed training device: cpu
Running on CPU.


## 5. Download Market Data

This cell downloads OHLCV data from yfinance for the training period. The raw data is kept in memory so later cells can build features without downloading again.

In [15]:
raw_market_data = {}

for config in CONFIGS_TO_TRAIN:
    print(f"\n=== Downloading data for {config.name} ===")
    print(f"Tickers: {', '.join(config.tickers)}")
    raw_market_data[config.name] = download_ohlcv(config.tickers, TRAIN_START, TRAIN_END)
    for ticker, df in raw_market_data[config.name].items():
        print(f"{ticker}: {len(df)} rows, {df['date'].min().date()} to {df['date'].max().date()}")


=== Downloading data for pair_0050_2330 ===
Tickers: 0050.TW, 2330.TW
0050.TW: 1700 rows, 2018-01-02 to 2024-12-30
2330.TW: 1700 rows, 2018-01-02 to 2024-12-30

=== Downloading data for basket_0050_2330_2412 ===
Tickers: 0050.TW, 2330.TW, 2412.TW
0050.TW: 1700 rows, 2018-01-02 to 2024-12-30
2330.TW: 1700 rows, 2018-01-02 to 2024-12-30
2412.TW: 1700 rows, 2018-01-02 to 2024-12-30


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 6. Build Feature Frames

This converts the raw OHLCV data into the SMC and spread z-score features used by the PPO environment.

In [16]:
feature_data = {}

for config in CONFIGS_TO_TRAIN:
    feature_df = build_feature_frame(raw_market_data[config.name], config)
    feature_data[config.name] = feature_df
    print(f"{config.name}: {len(feature_df)} feature rows, {len(feature_df.columns)} columns")

feature_data[CONFIGS_TO_TRAIN[0].name].head()

pair_0050_2330: 1700 feature rows, 19 columns


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


basket_0050_2330_2412: 1700 feature rows, 29 columns


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Price,date,0050_TW_open,0050_TW_high,0050_TW_low,0050_TW_close,0050_TW_volume,0050_TW_PD_Pos,0050_TW_OB_Dist,0050_TW_FVG_Signal,2330_TW_open,2330_TW_high,2330_TW_low,2330_TW_close,2330_TW_volume,2330_TW_PD_Pos,2330_TW_OB_Dist,2330_TW_FVG_Signal,Spread_ZScore_0050_TW_2330_TW,Spread_ZScore
0,2018-01-02,20.537500,20.650000,20.537500,20.6500,14452796,0.5,0.0,0.0,231.5,232.5,231.0,232.5,18055269,0.5,0.0,0.0,0.0,0.0
1,2018-01-03,20.737499,20.862499,20.737499,20.8375,28785748,0.5,0.0,0.0,236.0,238.0,235.5,237.0,29308091,0.5,0.0,0.0,0.0,0.0
2,2018-01-04,20.875000,20.912500,20.799999,20.8750,22510260,0.5,0.0,1.0,240.0,240.0,236.5,239.5,29096613,0.5,0.0,1.0,0.0,0.0
3,2018-01-05,20.875000,20.950001,20.825001,20.9375,30467184,0.5,0.0,1.0,240.0,240.0,238.0,240.0,22438255,0.5,0.0,1.0,0.0,0.0
4,2018-01-08,20.950001,21.037500,20.924999,21.0250,20758444,0.5,0.0,1.0,242.0,242.5,240.5,242.0,20233692,0.5,0.0,1.0,0.0,0.0


## 7. Training Function

In [17]:
def train_one_model(config):
    print(f"\n=== Training {config.name} ===")
    print(f"Tickers: {', '.join(config.tickers)}")
    print(f"Period: {TRAIN_START} to {TRAIN_END}")

    if config.name in feature_data:
        feature_df = feature_data[config.name]
    else:
        raw_frames = download_ohlcv(config.tickers, TRAIN_START, TRAIN_END)
        feature_df = build_feature_frame(raw_frames, config)
    print(f"Feature rows: {len(feature_df)}")

    env = DummyVecEnv(
        [
            lambda: TWTradingEnv(
                feature_df=feature_df,
                config=config,
                initial_balance=INITIAL_BALANCE,
            )
        ]
    )

    policy_kwargs = dict(net_arch=dict(pi=[128, 128], vf=[128, 128]))
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=2e-4,
        n_steps=2048,
        batch_size=128,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.005,
        target_kl=0.03,
        policy_kwargs=policy_kwargs,
        verbose=1,
        device=DEVICE,
    )

    model.learn(total_timesteps=TOTAL_TIMESTEPS)
    ensure_model_dir(config.model_path)
    model.save(config.model_path)
    print(f"Saved model: {config.model_path}")
    return model

## 8. Start Training

This cell may take a long time. It saves models to the paths defined in `PAIR_CONFIG` and `BASKET_CONFIG`.

In [18]:
trained_models = {}

for config in CONFIGS_TO_TRAIN:
    trained_models[config.name] = train_one_model(config)

trained_models.keys()


=== Training pair_0050_2330 ===
Tickers: 0050.TW, 2330.TW
Period: 2018-01-01 to 2024-12-31
Feature rows: 1700
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 552  |
|    iterations      | 1    |
|    time_elapsed    | 3    |
|    total_timesteps | 2048 |
-----------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------------
| time/                   |              |
|    fps                  | 470          |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0035420216 |
|    clip_fraction        | 0.0241       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | -1.28        |
|    learning_rate        | 0.0002       |
|    loss                 | -0.00128     |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00167     |
|    std                  | 1            |
|    value_loss           | 0.00742      |
------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------------
| time/                   |              |
|    fps                  | 480          |
|    iterations           | 3            |
|    time_elapsed         | 12           |
|    total_timesteps      | 6144         |
| train/                  |              |
|    approx_kl            | 0.0024858934 |
|    clip_fraction        | 0.0151       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.43        |
|    explained_variance   | 0.128        |
|    learning_rate        | 0.0002       |
|    loss                 | -0.00613     |
|    n_updates            | 20           |
|    policy_gradient_loss | -0.00259     |
|    std                  | 1.01         |
|    value_loss           | 0.00559      |
------------------------------------------


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| time/                   |             |
|    fps                  | 461         |
|    iterations           | 245         |
|    time_elapsed         | 1087        |
|    total_timesteps      | 501760      |
| train/                  |             |
|    approx_kl            | 0.011365718 |
|    clip_fraction        | 0.124       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.35       |
|    explained_variance   | 0.538       |
|    learning_rate        | 0.0002      |
|    loss                 | -0.000953   |
|    n_updates            | 2440        |
|    policy_gradient_loss | -0.00752    |
|    std                  | 0.531       |
|    value_loss           | 0.0411      |
-----------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved model: model/saved/ppo_model_basket.zip


dict_keys(['pair_0050_2330', 'basket_0050_2330_2412'])

## 9. Export Models to ONNX

This section exports the trained PPO models to the ONNX format, allowing them to be used in other environments or with ONNX runtime for deployment.

In [33]:
import torch # Ensure torch is imported for ONNX export
import torch.nn as nn # For the wrapper

for config in CONFIGS_TO_TRAIN:
    model = trained_models[config.name]

    # Re-create the environment to get the observation space shape
    # This is necessary because the model object itself doesn't directly expose env.observation_space
    # but we need it for the input_shape of ONNX export.
    feature_df = feature_data[config.name]
    env = TWTradingEnv(
        feature_df=feature_df,
        config=config,
        initial_balance=INITIAL_BALANCE,
    )
    # The input shape for ONNX export needs to be a tuple (batch_size, obs_dim).
    # Since the model expects a single observation at a time for inference, batch_size is 1.
    input_shape = (1, env.observation_space.shape[0])

    # Define the ONNX output path
    onnx_path = config.model_path.replace('.zip', '.onnx')
    ensure_model_dir(onnx_path)

    print(f"\nExporting {config.name} to ONNX: {onnx_path}")

    # Create a dummy input tensor for ONNX export
    # Ensure it's on the correct device (CPU or CUDA, as determined by DEVICE global)
    dummy_input = torch.randn(input_shape).to(DEVICE)

    # Set the model to evaluation mode
    model.policy.eval()

    # Define a wrapper module to expose only the deterministic prediction
    class SB3PolicyPredictor(nn.Module):
        def __init__(self, policy_module):
            super().__init__()
            # Extract the relevant sub-modules for deterministic action prediction
            self.features_extractor = policy_module.features_extractor
            self.mlp_extractor = policy_module.mlp_extractor
            self.action_net = policy_module.action_net

        def forward(self, obs: torch.Tensor) -> torch.Tensor:
            # Manually trace the deterministic action prediction path
            # 1. Extract features from observation
            features = self.features_extractor(obs)
            # 2. Pass features through the actor's MLP to get latent_pi
            latent_pi = self.mlp_extractor.forward_actor(features)
            # 3. Get the mean actions from the action network
            mean_actions = self.action_net(latent_pi)
            return mean_actions

    # Create an instance of our wrapper
    predictor = SB3PolicyPredictor(model.policy)

    input_names = ['obs']
    # Our wrapper only outputs the 'action'
    output_names = ['action']

    try:
        torch.onnx.export(
            predictor, # Export the wrapper module
            dummy_input,
            onnx_path,
            verbose=True, # Set to True for more detailed output during export
            input_names=input_names,
            output_names=output_names,
            opset_version=18, # Use opset_version 18
            dynamic_shapes={ # Specify dynamic batch size only for the input 'obs'
                'obs': {0: 'batch_size'}
            }
        )
        print(f"Successfully exported {config.name} model to {onnx_path}")
    except Exception as e:
        print(f"Error exporting {config.name} model to ONNX: {e}")


Exporting pair_0050_2330 to ONNX: model/saved/ppo_model_pair.onnx


/tmp/ipykernel_5874/2966189933.py:60: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SB3PolicyPredictor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SB3PolicyPredictor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/tmp/ipykernel_5874/2966189933.py:60: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Successfully exported pair_0050_2330 model to model/saved/ppo_model_pair.onnx

Exporting basket_0050_2330_2412 to ONNX: model/saved/ppo_model_basket.onnx


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[torch.onnx] Obtain model graph for `SB3PolicyPredictor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SB3PolicyPredictor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Successfully exported basket_0050_2330_2412 model to model/saved/ppo_model_basket.onnx


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 10. Load ONNX Model and Perform Inference

This section loads the exported ONNX model and uses it to perform inference on the last 20% of the data, simulating trading and evaluating performance.

In [34]:
# Install onnxruntime if not already installed
%pip install -q onnxruntime

In [35]:
import onnxruntime as ort

# Define the configuration for the model we want to test
# Assuming '0050' refers to the PAIR_CONFIG model
config_to_test = PAIR_CONFIG

# Define the ONNX path
onnx_model_path = config_to_test.model_path.replace('.zip', '.onnx')

# Load the ONNX model
print(f"Loading ONNX model from: {onnx_model_path}")

try:
    # Create an ONNX Runtime session
    # Run on CPU as DEVICE is 'cpu'
    ort_session = ort.InferenceSession(onnx_model_path, providers=['CPUExecutionProvider'])
    print("ONNX model loaded successfully.")
except Exception as e:
    print(f"Error loading ONNX model: {e}")
    ort_session = None

if ort_session:
    # Prepare the data for inference (last 20% of the feature data)
    feature_df = feature_data[config_to_test.name]
    total_rows = len(feature_df)
    inference_start_index = int(total_rows * 0.8)
    inference_df = feature_df.iloc[inference_start_index:].reset_index(drop=True)

    print(f"\nPrepared inference data from index {inference_start_index} (last 20%): {len(inference_df)} rows")

    # Create a new environment for inference
    eval_env = TWTradingEnv(
        feature_df=inference_df,
        config=config_to_test,
        initial_balance=INITIAL_BALANCE,
    )

    obs, _ = eval_env.reset()
    done = False
    episode_reward = 0
    inference_history = []

    print("\nStarting inference...")
    while not done:
        # Convert observation to float32 and add batch dimension (1, obs_dim)
        obs_tensor = np.array(obs, dtype=np.float32).reshape(1, -1)

        # Get model outputs from ONNX runtime
        # The input name is 'obs' as defined during export
        # The output names are 'action', 'value', 'log_prob'
        ort_inputs = {ort_session.get_inputs()[0].name: obs_tensor}
        ort_outputs = ort_session.run(None, ort_inputs)

        # Extract action from the outputs. It's the first output.
        action = ort_outputs[0][0] # Assuming action is the first output and we need the first item in batch

        obs, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated
        episode_reward += reward

        # Collect history for analysis
        inference_history.append(eval_env.history[-1])

    print("Inference completed.")
    print(f"Total reward: {episode_reward}")

    # Analyze results
    inference_history_df = pd.DataFrame(inference_history)
    metrics = calculate_metrics(inference_history_df, INITIAL_BALANCE)

    print("\nInference Performance Metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value:.4f}")

    print("\nInference History (first 5 rows):")
    display(inference_history_df.head())

    print("\nInference History (last 5 rows):")
    display(inference_history_df.tail())

Loading ONNX model from: model/saved/ppo_model_pair.onnx
ONNX model loaded successfully.

Prepared inference data from index 1360 (last 20%): 340 rows

Starting inference...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Inference completed.
Total reward: 1.1526766350108977

Inference Performance Metrics:
  cumulative_return: 0.5291
  sharpe_ratio: 1.3210
  max_drawdown: -0.2402

Inference History (first 5 rows):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,date,raw_action,net_worth,pnl,pnl_rate,transaction_cost,drawdown,reward,failed_orders,raw_action_0050.TW,weight_0050.TW,weight_2330.TW
0,2023-08-10,0.836580,999320.488155,-679.511845,-0.000680,679.511845,0.000680,-2.718047e-03,0,0.836580,0.795291,0.0
1,2023-08-11,1.161151,998559.526194,-760.961961,-0.000761,132.213100,0.001440,-1.654227e-03,0,1.161151,0.950126,0.0
2,2023-08-14,1.188992,985037.553336,-13521.972858,-0.013541,0.470654,0.014962,-2.704442e-02,0,1.188992,0.950000,0.0
3,2023-08-15,1.142877,986540.661022,1503.107686,0.001526,0.140256,0.013459,-1.423861e-07,0,1.142877,0.950000,0.0
4,2023-08-16,1.122733,981655.261922,-4885.399100,-0.004952,0.208728,0.018345,-9.771010e-03,0,1.122733,0.950000,0.0



Inference History (last 5 rows):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,date,raw_action,net_worth,pnl,pnl_rate,transaction_cost,drawdown,reward,failed_orders,raw_action_0050.TW,weight_0050.TW,weight_2330.TW
334,2024-12-23,-2.289973,1.515723e+06,60111.571858,4.129644e-02,11.596047,0.131792,3.543928e-02,0,-2.289973,0.0,0.950007
335,2024-12-24,-2.291606,1.515723e+06,-0.042468,-2.801806e-08,0.042468,0.131792,-1.310567e-07,0,-2.291606,0.0,0.950000
336,2024-12-25,-2.317322,1.522388e+06,6665.089715,4.397300e-03,1.285099,0.127975,1.132450e-02,0,-2.317322,0.0,0.950001
337,2024-12-26,-2.313890,1.522388e+06,-0.004706,-3.091429e-09,0.004706,0.127975,-1.656378e-08,0,-2.313890,0.0,0.950000
338,2024-12-27,-2.329868,1.529052e+06,6663.548307,4.377036e-03,1.284664,0.124158,1.211418e-02,0,-2.329868,0.0,0.950001


### Install missing ONNX dependencies

In [21]:
%pip install -q onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 14.5 MB/s eta 0:00:00
